Phase A: Data Engineering & Preprocessing Pipeline

Overview
The objective of Phase A is to transform the raw MPIIGaze dataset—a professional research dataset containing 213,659 images of 15 subjects—into a format optimized for Deep Learning. This stage focuses on extracting nested data, normalizing eye images, and converting 3D gaze vectors into 2D rotational angles.

In [8]:
import os
import cv2
import numpy as np
import pandas as pd
import scipy.io as sio
from tqdm import tqdm

In [ ]:

def vector_to_angles(vectors):
    vectors = np.array(vectors)
    x, y, z = vectors[:, 0], vectors[:, 1], vectors[:, 2]
    pitch = np.arcsin(-y)
    yaw = np.arctan2(-x, -z)
    return np.stack([pitch, yaw], axis=1)

def process_proctor_data_chunked(dataset_root, output_dir, chunk_size=20000):
    os.makedirs(output_dir, exist_ok=True)
    
    subjects = [s for s in os.listdir(dataset_root) if s.startswith('p')]
    all_images = []
    all_labels = []
    chunk_count = 0
    total_processed = 0

    print(f"Processing {len(subjects)} subjects in chunks of {chunk_size}...")

    for subject in subjects:
        subject_path = os.path.join(dataset_root, subject)
        for root, dirs, files in os.walk(subject_path):
            for file in files:
                if file.endswith('.mat'):
                    try:
                        data_mat = sio.loadmat(os.path.join(root, file))
                
                        data = data_mat['data']
                        eye_data = data['left'][0][0]
                        image_v = eye_data['image'][0][0]
                        gaze_v = eye_data['gaze'][0][0]

                        for i in range(image_v.shape[0]):
                            img = image_v[i].reshape(36, 60)
                            img = cv2.equalizeHist(img.astype(np.uint8))
                            img = cv2.resize(img, (64, 64))
                            
                            all_images.append(img.reshape(64, 64, 1))
                            all_labels.append(gaze_v[i].flatten())
                            total_processed += 1

                            if len(all_images) >= chunk_size:
                                chunk_count += 1
                                save_path_x = os.path.join(output_dir, f'X_chunk_{chunk_count}.npy')
                                save_path_y = os.path.join(output_dir, f'Y_chunk_{chunk_count}.npy')
                                
                                
                                Y_pitch_yaw = vector_to_angles(np.array(all_labels))
                                
                                np.save(save_path_x, np.array(all_images, dtype=np.uint8))
                                np.save(save_path_y, Y_pitch_yaw.astype(np.float32))
                                
                        
                                all_images = []
                                all_labels = []
                                print(f"Saved chunk {chunk_count} ({total_processed} total images processed)")

                    except Exception:
                        continue

    if all_images:
        chunk_count += 1
        Y_pitch_yaw = vector_to_angles(np.array(all_labels))
        np.save(os.path.join(output_dir, f'X_chunk_{chunk_count}.npy'), np.array(all_images, dtype=np.uint8))
        np.save(os.path.join(output_dir, f'Y_chunk_{chunk_count}.npy'), Y_pitch_yaw.astype(np.float32))

    print(f"\nPhase A Complete! {total_processed} images saved across {chunk_count} chunks.")
    print(f"Location: {output_dir}")


dataset_path = r'./mpii_data/MPIIGaze/Data/Normalized' # Path we found earlier
output_path = './processed_chunks'
process_proctor_data_chunked(dataset_path, output_path)

Processing 15 subjects in chunks of 20000...
Saved chunk 1 (20000 total images processed)
Saved chunk 2 (40000 total images processed)
Saved chunk 3 (60000 total images processed)
Saved chunk 4 (80000 total images processed)
Saved chunk 5 (100000 total images processed)
Saved chunk 6 (120000 total images processed)
Saved chunk 7 (140000 total images processed)
Saved chunk 8 (160000 total images processed)
Saved chunk 9 (180000 total images processed)
Saved chunk 10 (200000 total images processed)

Phase A Complete! 213658 images saved across 11 chunks.
Location: ./processed_chunks


Mathematical Transformation: 3D to 2D GazeRaw gaze data is provided as a 3D unit vector (x, y, z). 
To reduce the model's complexity, we convert these into Spherical Coordinates:Pitch (theta): Vertical rotation (looking up/down).Yaw (phi): Horizontal rotation (looking left/right).
Equations:Pitch = arcsin(-y) and Yaw = arctan2(-x, -z)

Hardware Optimization: Chunked Serialization
Working with an 8GB RAM constraint, loading the entire dataset would cause system failure. I implemented a Chunked Data Pipeline:

Buffer Management: Images are processed in batches of 20,000.

Memory Clearing: Python’s list buffers are cleared after every disk write to maintain a stable memory footprint.

Serialized Storage: Data is stored as binary .npy files, using uint8 for images to save 75% more disk space compared to floating-point storage.

Phase B: Deep Learning Architecture & Model Training

In this phase, we transition from data engineering to Model Development. 
The objective is to build a Convolutional Neural Network (CNN) capable of performing Regression to predict continuous gaze coordinates (Pitch and Yaw).

Given the hardware constraint of 8GB RAM, the pipeline is engineered to utilize a chunk-based training approach to prevent memory overflow.

In [11]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import gc


In [9]:
def build_gaze_cnn():
    model = models.Sequential([
        layers.Input(shape=(64, 64, 1)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(2) 
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), 
                  loss='mse', metrics=['mae'])
    return model

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,451,080 (13.16 MB)

 Trainable params: 1,150,210 (4.39 MB)

 Non-trainable params: 448 (1.75 KB)

 Optimizer params: 2,300,422 (8.78 MB)

In [ ]:

chunk_dir = 'processed_chunks' 
tf.keras.backend.clear_session()

x_files = sorted(glob.glob(os.path.join(chunk_dir, 'X_chunk_*.npy')))
y_files = sorted(glob.glob(os.path.join(chunk_dir, 'Y_chunk_*.npy')))

if len(x_files) == 0:
    print(f"!!! CRITICAL ERROR !!!")
    print(f"No .npy files found in the folder: '{os.path.abspath(chunk_dir)}'")
    print("Please check if you named your folder differently in Phase A.")
else:
    print(f"Success! Found {len(x_files)} data chunks.")
    model = build_gaze_cnn()
    
    EPOCHS = 10
    BATCH_SIZE = 32

    for epoch in range(EPOCHS):
        epoch_losses = []
        print(f"\n--- EPOCH {epoch+1}/{EPOCHS} ---")
        
        for f_idx in range(len(x_files)):
            X_chunk = np.load(x_files[f_idx]).astype('float32')
            Y_chunk = np.load(y_files[f_idx]).astype('float32')
            
            if X_chunk.ndim == 3:
                X_chunk = np.expand_dims(X_chunk, axis=-1)
                
            history = model.fit(X_chunk, Y_chunk, batch_size=BATCH_SIZE, 
                                epochs=1, verbose=0, shuffle=True)
            
            epoch_losses.append(history.history['loss'][0])
            print(f" Chunk {f_idx+1}/{len(x_files)} | Loss: {history.history['loss'][0]:.6f}", end='\r')
            
            del X_chunk, Y_chunk
            gc.collect() 
        
        avg_loss = sum(epoch_losses) / len(epoch_losses)
        print(f"\n>> Epoch {epoch+1} Average Loss: {avg_loss:.6f}")


    model.save('proctor_gaze_model.h5')
    print("\nModel saved successfully as 'proctor_gaze_model.h5'!")

Success! Found 11 data chunks.

--- EPOCH 1/10 ---
 Chunk 11/11 | Loss: 0.008216
>> Epoch 1 Average Loss: 0.016668

--- EPOCH 2/10 ---
 Chunk 11/11 | Loss: 0.008696
>> Epoch 2 Average Loss: 0.007333

--- EPOCH 3/10 ---
 Chunk 11/11 | Loss: 0.006845
>> Epoch 3 Average Loss: 0.006053

--- EPOCH 4/10 ---
 Chunk 11/11 | Loss: 0.006371
>> Epoch 4 Average Loss: 0.005315

--- EPOCH 5/10 ---
 Chunk 11/11 | Loss: 0.006039
>> Epoch 5 Average Loss: 0.005110

--- EPOCH 6/10 ---
 Chunk 11/11 | Loss: 0.006048
>> Epoch 6 Average Loss: 0.004971

--- EPOCH 7/10 ---
 Chunk 11/11 | Loss: 0.005925
>> Epoch 7 Average Loss: 0.004805

--- EPOCH 8/10 ---
 Chunk 11/11 | Loss: 0.005911
>> Epoch 8 Average Loss: 0.004699

--- EPOCH 9/10 ---
 Chunk 11/11 | Loss: 0.005589
>> Epoch 9 Average Loss: 0.004570

--- EPOCH 10/10 ---



>> Epoch 10 Average Loss: 0.004495

Model saved successfully as 'proctor_gaze_model.h5'!


Training a deep learning model on the MPIIGaze dataset (213,659 images) poses a significant challenge for standard hardware. To prevent Out-Of-Memory (OOM) errors on an 8GB RAM system, we implemented a specialized "Segmented Ingestion Pipeline."

A. Segmented Data Ingestion
Instead of loading the entire 13GB dataset into memory, we utilized a Chunk-Based Loading strategy.
Serialized Chunks: In Phase A, the data was serialized into .npy files containing 20,000 samples each.
On-Demand Loading: During the training loop, only one chunk is active in the RAM at any given time.
Precision Casting: Images are stored as uint8 (0-255) on disk to save space and are only cast to float32 (decimals) during the active training step.

B. Explicit Memory Management
Python's default garbage collection is often too slow for high-velocity deep learning. We forced manual memory recovery:
Variable Deletion: After a chunk is processed by model.fit(), the X_chunk and Y_chunk variables are explicitly deleted using the del keyword.
Garbage Collection: We trigger gc.collect() to force the operating system to reclaim the freed memory addresses immediately.
Backend Clearing: We use tf.keras.backend.clear_session() at the start of the script to ensure no "ghost graphs" from previous failed runs are hogging the RAM.

C. Optimizer Memory Overhead
We utilized the Adam Optimizer, which is memory-intensive but computationally efficient.
State Buffers: For every 1.15M trainable parameters, Adam maintains two additional buffers (Momentum and Velocity), totaling ~2.3M optimizer parameters.
Optimization: By keeping the batch size at 32, we ensured that the backpropagation gradients did not exceed the available VRAM/RAM buffer during the weight update step.
